# Fine-tune PP-OCRv5 on handwritten Indonesian nota

Self-contained: downloads its own data, builds its own dataset, writes its own training
config. Needs no other file in this repository.

```bash
pip install paddlepaddle-gpu==3.0.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu118/
pip install paddleocr pillow matplotlib numpy
```

PaddlePaddle must match your CUDA version and is installed separately — it is where setup
usually fails. Check the driver with `nvidia-smi` first.

**Two traps this notebook is built to avoid**, both documented in `docs/dataset-audit.html`:

1. Roboflow emits ~3 brightness-augmented copies of each receipt, each with its own COCO
   `image_id`. Splitting on `image_id` scatters copies of one nota across train, val *and*
   test — measured at **96.1% of validation crops contaminated**. We split on the source
   receipt and assert disjointness.
2. 274 of 738 label classes appear ≤3 times. The character dictionary is therefore built
   from **all** category names, not just those in train.


## Choose a GPU

This box is shared. Pick a card that is idle, and mask the others so a stray
`.to('cuda')` cannot land on somebody else's training run.


In [ ]:
# ── GPU selection ────────────────────────────────────────────────────────────
# Set this BEFORE torch or paddle is imported. Once either initialises CUDA the
# variable is ignored — if you have already run the imports, restart the kernel.
import os, subprocess

print(subprocess.run(
    ['nvidia-smi', '--query-gpu=index,name,utilization.gpu,memory.used,memory.total',
     '--format=csv'], capture_output=True, text=True).stdout)

GPU = 1          # which physical GPU to use.  None = leave all visible.

if GPU is not None:
    os.environ['CUDA_VISIBLE_DEVICES'] = str(GPU)
print('CUDA_VISIBLE_DEVICES =', os.environ.get('CUDA_VISIBLE_DEVICES', '<unset: all GPUs>'))
print('Inside this notebook that GPU is addressed as device 0.')

## 0 · Setup


In [3]:
import os, json, re, csv, math, random, shutil, subprocess, collections, itertools
from pathlib import Path
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

WORK = Path.home() / 'snaptok_work'          # datasets, PaddleOCR clone, checkpoints
RAW  = WORK / 'raw'                          # the COCO export
REC  = WORK / 'rec'                          # the recognition dataset we build
for d in (WORK, RAW, REC): d.mkdir(parents=True, exist_ok=True)

PADDLEOCR_TAG = 'v3.7.0'                     # pinned: the config schema moves between releases
PRETRAIN_URL  = ('https://paddle-model-ecology.bj.bcebos.com/paddlex/'
                 'official_pretrained_model/PP-OCRv5_mobile_rec_pretrained.pdparams')

def load_env(name, default=''):
    """Read a variable from the environment, falling back to a .env file
    found in this directory or any parent. Keeps secrets out of the notebook."""
    val = os.environ.get(name, '')
    if not val:
        for d in [Path.cwd(), *Path.cwd().parents]:
            f = d / '.env'
            if f.exists():
                for line in f.read_text().splitlines():
                    line = line.strip()
                    if line and not line.startswith('#') and '=' in line:
                        k, v = line.split('=', 1)
                        if k.strip() == name:
                            val = v.strip().strip('"').strip("'")
                break
    return val or default

ROBOFLOW_KEY = load_env('ROBOFLOW_API_KEY')
print('ROBOFLOW_API_KEY:', 'found' if ROBOFLOW_KEY else 'NOT SET — see .env.example')
WORKSPACE, PROJECT, VERSION = 'ocr-fqwqd', 'nota-pembelian', 9

def sh(cmd, cwd=None):
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout: print(line, end='')
    return p.wait()

print('work dir:', WORK)
sh('nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv')

ROBOFLOW_API_KEY: found
work dir: /Users/nafis/snaptok_work
/bin/sh: nvidia-smi: command not found


127

In [4]:
import paddle
print('paddle', paddle.__version__)
paddle.utils.run_check()
print('\npaddle sees', paddle.device.cuda.device_count(), 'GPU(s);',
      'compiled with CUDA:', paddle.device.is_compiled_with_cuda())

ModuleNotFoundError: No module named 'paddle'

## 1 · Download the corpus

~331 MB. Set `ROBOFLOW_API_KEY` in your environment first. Idempotent.


In [ ]:
import urllib.request

if (RAW / 'train').exists():
    print('already downloaded:', len(list((RAW / 'train').glob('*.jpg'))), 'images')
else:
    assert ROBOFLOW_KEY, 'set ROBOFLOW_API_KEY in your environment'
    api = (f'https://api.roboflow.com/{WORKSPACE}/{PROJECT}/{VERSION}/coco'
           f'?api_key={ROBOFLOW_KEY}')
    link = json.loads(urllib.request.urlopen(api).read())['export']['link']
    zip_path = RAW / 'export.zip'
    print('downloading ~331 MB ...')
    urllib.request.urlretrieve(link, zip_path)
    shutil.unpack_archive(str(zip_path), str(RAW))
    zip_path.unlink()
    print('done:', len(list((RAW / 'train').glob('*.jpg'))), 'images')

COCO = RAW / 'train' / '_annotations.coco.json'
coco = json.loads(COCO.read_text())
print(f"{len(coco['images'])} images · {len(coco['annotations']):,} annotations "
      f"· {len(coco['categories'])} categories")

## 2 · Build the recognition dataset

Crop every annotation box with 5% padding and write PaddleOCR label files.

Train keeps all augmented copies (legitimate augmentation); val and test keep exactly one
copy per receipt, so evaluation isn't three near-identical votes on the same image.


In [ ]:
SOURCE_RE = re.compile(r'^(.*?)_jpg\.rf\.')
PLACEHOLDER = {'objects', 'object'}

def source_id(file_name):
    """nota203_jpg.rf.93fc05d4....jpg -> nota203"""
    m = SOURCE_RE.match(file_name)
    return m.group(1) if m else file_name

def crop_with_padding(image, bbox, ratio=0.05):
    x, y, w, h = bbox
    px, py = w * ratio, h * ratio
    return image.crop((max(0, int(x - px)), max(0, int(y - py)),
                       min(image.width,  int(x + w + px)),
                       min(image.height, int(y + h + py))))

def split_sources(sources, writer_map, val_frac, test_frac, seed):
    """Group by writer when a map is given -- handwriting style, not receipt
    identity, is what breaks in deployment. Otherwise group by receipt."""
    rng = random.Random(seed)
    if writer_map:
        groups = collections.defaultdict(list)
        for s in sources:
            groups[writer_map.get(s, f'__unmapped__{s}')].append(s)
        keys = sorted(groups); rng.shuffle(keys)
        n_val, n_test = max(1, round(len(keys)*val_frac)), max(1, round(len(keys)*test_frac))
        buckets = {'val': keys[:n_val], 'test': keys[n_val:n_val+n_test],
                   'train': keys[n_val+n_test:]}
        return {k: {s for kk in v for s in groups[kk]} for k, v in buckets.items()}
    ordered = sorted(sources); rng.shuffle(ordered)
    n_val, n_test = round(len(ordered)*val_frac), round(len(ordered)*test_frac)
    return {'val': set(ordered[:n_val]), 'test': set(ordered[n_val:n_val+n_test]),
            'train': set(ordered[n_val+n_test:])}

In [ ]:
VAL_FRAC, TEST_FRAC, SEED = 0.15, 0.15, 42
WRITER_MAP = {}     # {'nota203': 'A', ...} -- see section 2b

images = {i['id']: i for i in coco['images']}
categories = {c['id']: c['name'] for c in coco['categories']}

# character dictionary from EVERY category, not just those in train
charset = {ch for name in categories.values() if name not in PLACEHOLDER
           for ch in str(name) if not ch.isspace()}
(REC / 'dict.txt').write_text('\n'.join(sorted(charset)) + '\n', encoding='utf-8')

by_source = collections.defaultdict(list)
for img in coco['images']:
    by_source[source_id(img['file_name'])].append(img)

splits = split_sources(list(by_source), WRITER_MAP, VAL_FRAC, TEST_FRAC, SEED)
for a, b in itertools.combinations(('train','val','test'), 2):
    overlap = splits[a] & splits[b]
    assert not overlap, f'LEAK: {len(overlap)} receipts in both {a} and {b}'

selected = {}
for split, sources in splits.items():
    picked = []
    for s in sorted(sources):
        copies = sorted(by_source[s], key=lambda i: i['file_name'])
        picked.extend(copies if split == 'train' else copies[:1])
    selected[split] = picked

annotations = collections.defaultdict(list)
for a in coco['annotations']:
    annotations[a['image_id']].append(a)

counts, skipped = collections.Counter(), collections.Counter()
for split, imgs in selected.items():
    (REC / split).mkdir(parents=True, exist_ok=True)
    records = []
    for img in imgs:
        path = RAW / 'train' / img['file_name']
        if not path.exists():
            skipped['missing_image'] += 1; continue
        with Image.open(path) as handle:
            page = handle.convert('RGB')
            for ann in annotations[img['id']]:
                text = str(categories.get(ann['category_id'], '')).strip()
                if not text or text in PLACEHOLDER:
                    skipped['placeholder'] += 1; continue
                crop = crop_with_padding(page, ann['bbox'])
                if crop.width < 4 or crop.height < 4:
                    skipped['degenerate'] += 1; continue
                name = f"{img['id']}_{ann['id']}.jpg"
                crop.save(REC / split / name, quality=95)
                records.append((f'{split}/{name}', text))
    with (REC / f'{split}_rec.txt').open('w', encoding='utf-8') as fh:
        for rel, text in records: fh.write(f'{rel}\t{text}\n')
    counts[split] = len(records)

manifest = {'seed': SEED, 'grouped_by': 'writer' if WRITER_MAP else 'source_receipt',
            'receipts': {k: sorted(v) for k, v in splits.items()},
            'crops': dict(counts), 'charset_size': len(charset)}
(REC / 'split_manifest.json').write_text(json.dumps(manifest, indent=2))

print(f"grouping       : {manifest['grouped_by']}")
print(f'character dict : {len(charset)} chars')
for s in ('train','val','test'):
    print(f'{s:<7} {len(splits[s]):>4} receipts  {len(selected[s]):>4} images  {counts[s]:>6} crops')
if skipped: print('skipped        :', dict(skipped))
if not WRITER_MAP:
    print('\nNOTE: grouped by receipt, not writer. The same 2-3 hands appear in every')
    print('      split, so val accuracy overstates real performance.')

### 2a · Verify the split independently

Re-derives the guarantee from the COCO file rather than trusting the code above, by
tracing every crop back to its source receipt.


In [ ]:
src = {i['id']: source_id(i['file_name']) for i in coco['images']}
for split in ('train','val','test'):
    ids = {int(l.split('/')[1].split('_')[0]) for l in open(REC / f'{split}_rec.txt')}
    stray = {src[i] for i in ids} - set(manifest['receipts'][split])
    assert not stray, f'{split}: {len(stray)} crops from outside its split'
print('verified: every crop traces to a receipt inside its own split')

rows = [l.rstrip('\n').split('\t') for l in open(REC / 'train_rec.txt')]
sample = random.Random(0).sample(rows, 24)
fig, axes = plt.subplots(4, 6, figsize=(15, 6))
for ax, (rel, text) in zip(axes.ravel(), sample):
    ax.imshow(Image.open(REC / rel)); ax.set_title(text, fontsize=11); ax.axis('off')
plt.tight_layout(); plt.show()
print('numeric labels: %.0f%%' % (100 * sum(1 for _, t in rows if t.isdigit()) / len(rows)))

### 2b · Writer-held-out split (optional, higher value)

The corpus has only two or three distinct writers. Grouping by receipt still puts the same
handwriting in every split, so the number above flatters the model. Label the writers and
re-run section 2 with `WRITER_MAP` filled in:

```python
WRITER_MAP = {'nota001': 'A', 'nota002': 'A', 'nota003': 'B', ...}
```

Roughly an hour of eyeballing 440 receipts, and it converts your headline accuracy from
flattering to honest.


## 3 · Write the training config

Derived from PaddleOCR `configs/rec/PP-OCRv5/PP-OCRv5_mobile_rec.yml` at the pinned tag.
Every deviation from upstream is marked `# CHANGED` with its reason.

The important one: we keep the **upstream character dictionary**. Swapping in our 36-char
dict changes the CTC head's output shape, so the pretrained head is silently skipped and
retrains from scratch — a bad trade against 12.6k crops. Our charset is a subset of theirs.


In [ ]:
CONFIG_YAML = r"""# PP-OCRv5 mobile recognition, fine-tuned on handwritten Indonesian nota.
#
# Derived from PaddleOCR configs/rec/PP-OCRv5/PP-OCRv5_mobile_rec.yml (v3.7.0).
# Only the marked lines deviate from upstream; everything else is left at the
# authors' defaults deliberately, so that anything we change is a change we can
# justify and ablate.
#
# Baseline = condition 1 of the ablation ladder: real crops only, clean
# source-grouped split. This is the honest number the notebook never produced.

Global:
  model_name: PP-OCRv5_mobile_rec
  debug: false
  use_gpu: true
  epoch_num: 100                      # CHANGED 75->100: only ~12.6k crops, epochs are cheap
  log_smooth_window: 20
  print_batch_step: 20
  save_model_dir: ./output/nota_rec_v5_mobile
  save_epoch_step: 10
  eval_batch_step: [0, 200]           # CHANGED [0,2000]: 2000 steps is ~10 epochs at this size
  cal_metric_during_train: true
  pretrained_model: ./pretrain/PP-OCRv5_mobile_rec_pretrained.pdparams   # CHANGED
  checkpoints:
  save_inference_dir:
  use_visualdl: false
  infer_img: doc/imgs_words/ch/word_1.jpg
  # CHANGED: keep the FULL upstream dictionary rather than our 36-char dict.
  # Swapping in a 36-char dict changes the CTC head's output shape, so the
  # pretrained head is silently skipped and retrains from scratch -- a bad
  # trade against only 12.6k crops. Our charset is a subset of this one, so
  # nothing is lost. Restricting the output alphabet belongs at decode time.
  # Ablation: rerun with character_dict_path=data/rec/dict.txt and compare.
  character_dict_path: ./ppocr/utils/dict/ppocrv5_dict.txt
  max_text_length: &max_text_length 25
  infer_mode: false
  use_space_char: true
  distributed: false                  # CHANGED true->false: single A100
  save_res_path: ./output/nota_rec_v5_mobile/predicts.txt
  d2s_train_image_shape: [3, 48, 320]

Optimizer:
  name: Adam
  beta1: 0.9
  beta2: 0.999
  lr:
    name: Cosine
    learning_rate: 0.0001             # CHANGED 5e-4->1e-4: fine-tuning, not training from scratch
    warmup_epoch: 5
  regularizer:
    name: L2
    factor: 3.0e-05

Architecture:
  model_type: rec
  algorithm: SVTR_LCNet
  Transform:
  Backbone:
    name: PPLCNetV3
    scale: 0.95
  Head:
    name: MultiHead
    head_list:
      - CTCHead:
          Neck:
            name: svtr
            dims: 120
            depth: 2
            hidden_dims: 120
            kernel_size: [1, 3]
            use_guide: True
          Head:
            fc_decay: 0.00001
      - NRTRHead:
          nrtr_dim: 384
          max_text_length: *max_text_length

Loss:
  name: MultiLoss
  loss_config_list:
    - CTCLoss:
    - NRTRLoss:

PostProcess:
  name: CTCLabelDecode

Metric:
  name: RecMetric
  main_indicator: acc

Train:
  dataset:
    name: MultiScaleDataSet
    ds_width: false
    data_dir: ./data/rec/                          # CHANGED
    ext_op_transform_idx: 1
    label_file_list:
    - ./data/rec/train_rec.txt                     # CHANGED
    transforms:
    - DecodeImage:
        img_mode: BGR
        channel_first: false
    - RecConAug:
        # Upstream default prob 0.5. Our crops are single words, so heavy
        # concatenation trains on a length distribution we never see at
        # inference. Lowered, but this is an ablation knob, not a fact.
        prob: 0.3                                  # CHANGED
        ext_data_num: 2
        image_shape: [48, 320, 3]
        max_text_length: *max_text_length
    - RecAug:
    - MultiLabelEncode:
        gtc_encode: NRTRLabelEncode
    - KeepKeys:
        keep_keys:
        - image
        - label_ctc
        - label_gtc
        - length
        - valid_ratio
  sampler:
    name: MultiScaleSampler
    scales: [[320, 32], [320, 48], [320, 64]]
    first_bs: &bs 64                               # CHANGED 128->64: more updates on a small set
    fix_bs: false
    divided_factor: [8, 16]
    is_training: True
  loader:
    shuffle: true
    batch_size_per_card: *bs
    drop_last: true
    num_workers: 8

Eval:
  dataset:
    name: SimpleDataSet
    data_dir: ./data/rec/                          # CHANGED
    label_file_list:
    - ./data/rec/val_rec.txt                       # CHANGED
    transforms:
    - DecodeImage:
        img_mode: BGR
        channel_first: false
    - MultiLabelEncode:
        gtc_encode: NRTRLabelEncode
    - RecResizeImg:
        image_shape: [3, 48, 320]
    - KeepKeys:
        keep_keys:
        - image
        - label_ctc
        - label_gtc
        - length
        - valid_ratio
  loader:
    shuffle: false
    drop_last: false
    batch_size_per_card: 64                        # CHANGED
    num_workers: 4
"""

PADDLE_DIR = WORK / 'PaddleOCR'
CONFIG_PATH = PADDLE_DIR / 'configs' / 'rec' / 'nota_rec_v5_mobile.yml'
print(CONFIG_YAML[:400], '...')

## 4 · Fine-tune

Clones PaddleOCR at the pinned tag, fetches pretrained weights, links the dataset, trains.

**Hours, not minutes.** If your session might drop, run it from a terminal instead:
`nohup python tools/train.py -c configs/rec/nota_rec_v5_mobile.yml > train.log 2>&1 &`


In [ ]:
if not PADDLE_DIR.exists():
    sh(f'git clone --depth 1 --branch {PADDLEOCR_TAG} '
       f'https://github.com/PaddlePaddle/PaddleOCR.git {PADDLE_DIR}')

(PADDLE_DIR / 'pretrain').mkdir(exist_ok=True)
weights = PADDLE_DIR / 'pretrain' / Path(PRETRAIN_URL).name
if not weights.exists():
    print('fetching pretrained weights ...')
    urllib.request.urlretrieve(PRETRAIN_URL, weights)
print('weights:', weights.stat().st_size // 2**20, 'MB')

(PADDLE_DIR / 'data').mkdir(exist_ok=True)
link = PADDLE_DIR / 'data' / 'rec'
if not link.exists(): link.symlink_to(REC)

CONFIG_PATH.write_text(CONFIG_YAML)
print('config ->', CONFIG_PATH)

In [ ]:
rc = sh('python tools/train.py -c configs/rec/nota_rec_v5_mobile.yml', cwd=PADDLE_DIR)
print('exit', rc)

## 5 · Export and evaluate

Test split — never seen during training or model selection.


In [ ]:
sh('python tools/export_model.py -c configs/rec/nota_rec_v5_mobile.yml '
   '-o Global.pretrained_model=./output/nota_rec_v5_mobile/best_accuracy '
   'Global.save_inference_dir=./output/nota_rec_v5_mobile_infer/', cwd=PADDLE_DIR)

sh('python tools/eval.py -c configs/rec/nota_rec_v5_mobile.yml '
   '-o Global.checkpoints=./output/nota_rec_v5_mobile/best_accuracy '
   'Eval.dataset.label_file_list=[./data/rec/test_rec.txt]', cwd=PADDLE_DIR)

### 5a · Digit-stratified error

PaddleOCR reports one accuracy figure. That hides what matters: a wrong letter in a product
name is cosmetic, a wrong digit in a price silently corrupts inventory.


In [ ]:
from paddleocr import TextRecognition

INFER = PADDLE_DIR / 'output' / 'nota_rec_v5_mobile_infer'
rec = TextRecognition(model_dir=str(INFER))

def cer(pred, gold):
    """Levenshtein distance normalised by reference length."""
    prev = list(range(len(gold) + 1))
    for i, p in enumerate(pred, 1):
        cur = [i]
        for j, g in enumerate(gold, 1):
            cur.append(min(prev[j] + 1, cur[j-1] + 1, prev[j-1] + (p != g)))
        prev = cur
    return prev[-1] / max(len(gold), 1)

test_rows = [l.rstrip('\n').split('\t') for l in open(REC / 'test_rec.txt')]
preds = []
for i in range(0, len(test_rows), 256):
    batch = [str(REC / rel) for rel, _ in test_rows[i:i+256]]
    preds += [r.get('rec_text') or '' for r in rec.predict(batch)]

buckets, exact, worst = {'digits': [], 'letters': [], 'all': []}, 0, []
for (rel, gold), pred in zip(test_rows, preds):
    e = cer(pred, gold)
    buckets['all'].append(e)
    buckets['digits' if gold.isdigit() else 'letters'].append(e)
    exact += (pred == gold)
    if pred != gold: worst.append((e, gold, pred))

for k, v in buckets.items():
    if v: print(f'{k:<8} CER {sum(v)/len(v):6.2%}   n={len(v)}')
print(f'\nexact-match  {exact/len(test_rows):.2%}  ({exact}/{len(test_rows)} crops)')
print('\nworst misreads (gold -> predicted):')
for e, gold, pred in sorted(worst, reverse=True)[:15]:
    print(f"  {e:5.2f}  {gold!r:>12} -> {pred!r}{'   <-- DIGIT' if gold.isdigit() else ''}")

## 6 · What next

The exported model at `~/snaptok_work/PaddleOCR/output/nota_rec_v5_mobile_infer/` is what
the OCR service will load.

1. **Writer-held-out split** (section 2b) — until then this number is measured on the same
   few hands the model trained on.
2. **Synthetic handwriting.** The literature reports CER reductions of 13–36% from
   generated training data at roughly this corpus size. See `docs/run-a-research.html`.
3. **INT8 quantization** for CPU serving, and measure what accuracy it costs.
